# Silero Baseline Metrics

Evaluate Silero VAD on TORGO pilot subset.

Metrics:
- AUC (Area Under ROC Curve)
- Miss Rate (False Negative Rate)
- False Alarm Rate (False Positive Rate)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_auc_score, roc_curve

%matplotlib inline

## Load Manifest and Teacher Outputs

In [ ]:
manifest = pd.read_csv('../manifests/torgo_pilot.csv')
print(f"Total utterances: {len(manifest)}")

# Load cached teacher probabilities
teacher_dir = Path('../teacher_probs/')

all_probs = []
all_labels = []  # Placeholder - need ground truth labels

for _, row in manifest.iterrows():
    speaker_id = row['speaker_id']
    utt_id = row['utt_id']
    prob_file = teacher_dir / f"{speaker_id}_{utt_id}.npy"
    
    if prob_file.exists():
        probs = np.load(prob_file)
        all_probs.extend(probs)
        # TODO: Load corresponding ground truth labels
        # For now, use placeholder
        labels = np.random.randint(0, 2, size=len(probs))  # Placeholder!
        all_labels.extend(labels)

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

print(f"Total frames: {len(all_probs)}")

## Compute Metrics at Threshold 0.5

In [ ]:
threshold = 0.5
predictions = (all_probs >= threshold).astype(int)

# Confusion matrix components
tp = np.sum((predictions == 1) & (all_labels == 1))
fp = np.sum((predictions == 1) & (all_labels == 0))
tn = np.sum((predictions == 0) & (all_labels == 0))
fn = np.sum((predictions == 0) & (all_labels == 1))

# Metrics
miss_rate = fn / (fn + tp) if (fn + tp) > 0 else 0
far = fp / (fp + tn) if (fp + tn) > 0 else 0

print(f"Threshold: {threshold}")
print(f"Miss Rate: {miss_rate:.4f}")
print(f"False Alarm Rate: {far:.4f}")
print(f"")
print(f"Confusion Matrix:")
print(f"  TP: {tp}, FP: {fp}")
print(f"  FN: {fn}, TN: {tn}")

## ROC Curve and AUC

In [ ]:
fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
auc = roc_auc_score(all_labels, all_probs)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'AUC = {auc:.4f}')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Silero VAD on TORGO')
plt.legend()
plt.grid(True)
plt.show()

print(f"AUC: {auc:.4f}")

## Miss Rate vs False Alarm Tradeoff

In [ ]:
# Compute MR and FAR at different thresholds
thresholds = np.arange(0.1, 1.0, 0.05)
miss_rates = []
fars = []

for thresh in thresholds:
    preds = (all_probs >= thresh).astype(int)
    
    fn = np.sum((preds == 0) & (all_labels == 1))
    tp = np.sum((preds == 1) & (all_labels == 1))
    fp = np.sum((preds == 1) & (all_labels == 0))
    tn = np.sum((preds == 0) & (all_labels == 0))
    
    mr = fn / (fn + tp) if (fn + tp) > 0 else 0
    far = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    miss_rates.append(mr)
    fars.append(far)

plt.figure(figsize=(8, 6))
plt.plot(fars, miss_rates, 'b-o')
plt.xlabel('False Alarm Rate')
plt.ylabel('Miss Rate')
plt.title('DET Curve - Silero VAD on TORGO')
plt.grid(True)
plt.show()

## Per-Speaker Breakdown

In [ ]:
# TODO: Compute metrics per speaker for atypical vs control comparison
print("Per-speaker metrics will be computed here")

## Summary

| Metric | Value |
|--------|-------|
| AUC | {auc:.4f} |
| Miss Rate @ 0.5 | {miss_rate:.4f} |
| FAR @ 0.5 | {far:.4f} |

**Note**: These are baseline numbers for the pilot subset. Ground truth labels are needed for accurate evaluation.